# Strands 멀티 에이전트 시스템과 AgentCore 메모리 도구 (단기 메모리) - MemoryManager 사용

## 소개

이 노트북은 AWS AgentCore 메모리와 Strands 프레임워크를 사용하여 **공유 메모리를 갖춘 멀티 에이전트 시스템**을 구현하는 방법을 시연합니다. 이전 예제가 단일 에이전트 메모리에 초점을 맞춘 반면, 이 노트북에서는 여러 전문 에이전트가 공통 메모리 저장소에 접근하면서 함께 작업하는 방법을 탐구합니다.

**참고: 이 샘플은 원래 MemoryClient 대신 MemoryManager 및 MemorySessionManager를 사용하는 단기 메모리 샘플 버전입니다.**

## 튜토리얼 세부 정보

| 정보                | 세부 사항                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 사용 사례  | 여행 계획 어시스턴트                                                             |
| 에이전트 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Strands Agents, 도구를 통한 메모리 검색                   |
| 예제 복잡도         | 초급                                                                             |


학습 내용:

- 여러 에이전트가 접근할 수 있는 공유 메모리 리소스 설정 방법
- 자체 메모리 접근 권한을 가진 전문 에이전트를 도구로 생성
- 전문 에이전트에게 작업을 위임하는 코디네이터 에이전트 구현
- 여러 에이전트 상호작용 간 대화 맥락 유지

### 시나리오 맥락

이 예제에서는 다음으로 구성된 **여행 계획 시스템**을 만듭니다:
1. 항공 여행을 전문으로 하는 항공편 예약 어시스턴트
2. 숙박에 중점을 둔 호텔 예약 어시스턴트
3. 이러한 전문 에이전트에게 위임하는 여행 코디네이터

이 접근 방식은 복잡한 도메인을 동일한 메모리 저장소를 공유하는 전문 에이전트로 분할하는 방법을 보여줍니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항
- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

환경 설정과 공유 메모리 리소스 생성을 시작합니다!

## 1단계: 환경 설정
필요한 모든 라이브러리를 임포트하고 이 노트북이 작동하는 데 필요한 클라이언트를 정의합니다.

In [1]:
!pip install -qr requirements.txt

In [ ]:
import logging
import os
from datetime import datetime
from botocore.exceptions import ClientError
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# 메모리 관리 모듈 임포트
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

Amazon Bedrock 모델 및 AgentCore에 적절한 권한이 있는 리전과 역할을 정의합니다.

In [3]:
REGION = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## 2단계: 공유 메모리 생성
이 섹션에서는 전문 에이전트들 사이에서 공유될 메모리 리소스를 생성합니다.

In [ ]:
memory_manager = MemoryManager(region_name=REGION)

try:
    print("메모리 생성 중...")
    memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")

    # 메모리 리소스 생성
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 전략 없음
        description="Short-term memory for travel agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간
        memory_execution_role_arn=None,  # 단기 메모리에는 선택사항
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory.id
    logger.info(f"MemoryManager로 메모리 생성/검색 성공:")
    logger.info(f"   메모리 ID: {memory_id}")
    logger.info(f"   메모리 이름: {memory.name}")
    logger.info(f"   메모리 상태: {memory.status}")
except Exception as e:
    # 향상된 오류 보고로 메모리 생성 중 오류 처리
    logger.error(f"메모리 생성 실패: {e}")
    logger.error(f"오류 유형: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if 'memory_id' in locals():
        try:
            logger.info(f"부분적으로 생성된 메모리 정리 시도: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"메모리 정리 성공: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")
    
    # 원래 예외를 다시 발생시킴
    raise

### 멀티 에이전트 시스템을 위한 공유 메모리 이해

생성한 메모리 리소스는 여행 계획 시스템의 공유 지식 베이스 역할을 합니다. 모든 에이전트가 이 공통 메모리 저장소에서 읽고 쓰므로 다음이 가능합니다:

1. **지식 일관성**: 모든 에이전트가 동일한 정보로 작업
2. **맥락 보존**: 에이전트 전환 간에 대화 이력 유지
3. **전문화된 접근**: 각 에이전트가 고유한 actor_id를 가지지만 session_id를 공유

이 접근 방식을 통해 전문 에이전트가 전체 대화 맥락의 혜택을 받으면서 자신의 도메인에 집중할 수 있습니다.

## 3단계: 세션 매니저 초기화

이 섹션에서는 세션 기반 메모리 작업을 위한 MemorySessionManager를 소개하고 액터 및 세션을 관리하기 위한 MemorySession을 생성합니다.

In [ ]:
# 세션 메모리 매니저 초기화
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

logger.info(f"세션 매니저 초기화 완료 (메모리: {memory.id})")
logger.info(f"세션 매니저 유형: {type(session_manager)}")

## 4단계: 메모리 훅 프로바이더 생성

이 단계에서는 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트의 실행 생명주기의 특정 지점에서 실행되는 특수 함수입니다. 우리가 만드는 메모리 훅은 두 가지 주요 기능을 수행합니다:

1. **메모리 검색**: 사용자가 메시지를 보낼 때 관련 과거 대화를 자동으로 가져옵니다
2. **메모리 저장**: 에이전트가 응답한 후 새로운 대화를 저장합니다

**MemoryClient 버전과의 주요 차이점:**
- MemoryClient 대신 MemorySession 사용
- 튜플 대신 ConversationalMessage 객체 사용
- create_event() 대신 add_turns() 사용
- 타입 안전성을 위해 MessageRole enum 사용

In [ ]:
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_session: MemorySession, memory_id: str):
        self.memory_session = memory_session
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 이력 로드"""
        try:
            # 사전 구성된 메모리 세션 사용 (actor_id/session_id 불필요)
            recent_turns = self.memory_session.get_last_k_turns(k=5)
            
            if recent_turns:
                # 맥락을 위해 대화 이력 포맷
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # EventMessage 객체와 dict 형식 모두 처리
                        if hasattr(message, 'role') and hasattr(message, 'content'):
                            role = message['role']
                            content = message['content']
                        else:
                            role = message.get('role', 'unknown')
                            content = message.get('content', {}).get('text', '')
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                logger.info(f"메모리에서 가져온 맥락: {context}")
                
                # 에이전트의 시스템 프롬프트에 맥락 추가
                event.agent.system_prompt += f"\n\nRecent conversation history:\n{context}\n\nContinue the conversation naturally based on this context."
                logger.info(f"최근 {len(recent_turns)}개의 대화 턴 로드 완료")
            else:
                logger.info("이전 대화 이력이 없습니다")
                
        except Exception as e:
            logger.error(f"대화 이력 로드 실패: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """MemorySession을 사용하여 메모리에 메시지 저장"""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT
                
                # 메모리 세션 인스턴스 사용 (actor_id/session_id 전달 불필요)
                result = self.memory_session.add_turns(
                    messages=[ConversationalMessage(message_text, message_role)]
                )
                
                event_id = result['eventId']
                logger.info(f"메시지 저장 완료 (이벤트 ID: {event_id}, 역할: {message_role.value})")
                
        except Exception as e:
            logger.error(f"메모리 저장 오류: {e}")
            import traceback
            logger.error(f"전체 트레이스백: {traceback.format_exc()}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        # 메모리 훅 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 5단계: Strands 에이전트로 멀티 에이전트 아키텍처 생성
이 섹션에서는 항공편과 호텔 예약을 위한 전문 에이전트를 갖춘 멀티 에이전트 시스템을 만들며, 두 에이전트 모두 메모리 리소스에 대한 공유 접근 권한을 갖습니다.

In [ ]:
# 필요한 컴포넌트 임포트
from strands import Agent, tool

In [ ]:
# 각 전문 에이전트에 고유한 액터 ID를 생성하되 세션 ID는 공유
FLIGHT_ACTOR_ID = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
HOTEL_ACTOR_ID = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

### 메모리 접근 권한을 가진 전문 에이전트 생성

다음으로 전문 에이전트의 시스템 프롬프트를 정의합니다. 각 프롬프트에는 에이전트가 파싱할 수 있는 형식의 메모리 파라미터가 포함됩니다:

In [ ]:
# 호텔 예약 전문 에이전트의 시스템 프롬프트
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# 항공편 예약 전문 에이전트의 시스템 프롬프트
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

### 에이전트 도구 구현
이제 코디네이터 에이전트가 사용할 수 있는 도구로서 전문 에이전트를 구현합니다:

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        # 예약 어시스턴트를 위한 메모리 세션 생성
        memory_session = session_manager.create_memory_session(
            actor_id=FLIGHT_ACTOR_ID, 
            session_id=SESSION_ID
        )
        flight_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)
        
        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": FLIGHT_ACTOR_ID, "session_id": SESSION_ID}
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"항공편 예약 어시스턴트 오류: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        # 예약 어시스턴트를 위한 메모리 세션 생성
        memory_session = session_manager.create_memory_session(
            actor_id=HOTEL_ACTOR_ID, 
            session_id=SESSION_ID
        )

        hotel_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": HOTEL_ACTOR_ID, "session_id": SESSION_ID}
        )
        
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"호텔 예약 어시스턴트 오류: {str(e)}"

### 코디네이터 에이전트 생성

마지막으로 이러한 전문 도구들을 조율하는 메인 여행 계획 에이전트를 만듭니다:

In [ ]:
# 코디네이터 에이전트의 시스템 프롬프트
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [12]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

2025-10-10 23:05:10 - INFO - Found credentials in environment variables.


#### 멀티 에이전트 시스템이 준비되었습니다!!

## 에이전트를 테스트해봅시다.

여행 계획 시나리오로 멀티 에이전트 시스템을 테스트합니다:

In [ ]:
# "안녕하세요, LA에서 마드리드로 여행을 예약하고 싶습니다. 7월 1일부터 8월 2일까지입니다."라는 의미의 입력
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
# "지금은 항공편에만 집중하고 싶습니다. 직항, 중급, 도심, 수영장, 스탠다드 룸"이라는 의미의 입력
response = travel_agent("I would only like to focus on the flight at the moment. direct flimid-range, city center, pool, standard room")

## 메모리 지속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해 여행 에이전트의 새 인스턴스를 만들고 이전에 저장된 정보에 접근할 수 있는지 확인합니다:

In [ ]:
# 여행 에이전트의 새 인스턴스 생성
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

# 이전 대화에 대해 질문
# "이전에 이야기한 항공편에 대해 알려줄 수 있나요?"라는 의미의 입력
new_travel_agent("Can you remind me about flights talked about before?")

## 요약

이 노트북에서 시연한 내용:

1. 여러 에이전트를 위한 공유 메모리 리소스 생성 방법
2. 메모리 접근 권한을 가진 전문 에이전트를 도구로 구현하는 방법
3. 대화 맥락을 유지하면서 여러 에이전트 간 조율하는 방법
4. 서로 다른 에이전트 인스턴스 간에 메모리가 유지되는 방법

공유 메모리를 갖춘 이 멀티 에이전트 아키텍처는 일관된 사용자 경험을 유지하면서 전문 도메인을 처리할 수 있는 복잡한 대화형 AI 시스템을 구축하기 위한 강력한 접근 방식을 제공합니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다.

In [ ]:
# MemoryManager를 사용하여 메모리 리소스를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"메모리 삭제 완료: {memory_id}")
# except Exception as e:
#     logger.error(f"메모리 삭제 실패: {e}")